# Lesson 3: Inheritance for Data Workflows

## Learning Objectives

By the end of this lesson, you will be able to:
- Explain the concept of inheritance and its 'is-a' relationship.
- Create a child class that inherits from a parent (base) class.
- Use `super()` to call methods from the parent class, especially the constructor.
- Override parent class methods to provide specialized functionality in the child class.
- Design class hierarchies to reuse code effectively in a data science context.

## Why This Topic Matters

Imagine you've built a great `DataExtractor` class that knows how to connect to a source and pull data. Now, you need to extract data from different types of files: CSVs, JSON files, and Excel spreadsheets. The core logic of 'connecting' and 'providing data' is the same, but the specific implementation of reading the file is different for each type.

Without inheritance, you might be tempted to copy and paste the code into three separate classes (`CsvExtractor`, `JsonExtractor`, etc.). This leads to code duplication. If you find a bug in the shared logic, you have to fix it in three different places!

**Inheritance** solves this problem by allowing you to create a new class (a 'child') that adopts the attributes and methods of an existing class (a 'parent'). You can then add new features or change (override) existing ones. This promotes code reuse, reduces redundancy, and creates a logical structure for your project.

## Intuition-First Explanation: A Family Tree

Think of inheritance like a biological family tree.

- A **Parent** (also called a **Base Class** or **Superclass**) passes down its traits.
- A **Child** (also called a **Derived Class** or **Subclass**) **inherits** all of those traits.

For example:
- **Parent:** `Vehicle`. All vehicles have attributes like `speed` and methods like `accelerate()`.
- **Child:** `Car`. A `Car` **is a** `Vehicle`. It inherits `speed` and `accelerate()`. But it also adds its own specific attributes, like `num_doors`, and methods, like `open_trunk()`.
- **Child:** `Bicycle`. A `Bicycle` **is also a** `Vehicle`. It inherits `speed` and `accelerate()`, but its implementation of `accelerate()` (pedaling) is very different from a car's. It might also add a `ring_bell()` method.

The key phrase is **"is-a"**. A Car *is a* Vehicle. A Bicycle *is a* Vehicle. This relationship is the foundation of inheritance.

## Practical Example: A Base `DataLoader`

Let's create a general-purpose parent class for loading data. It will define the common structure and behavior that all data loaders should have.

In [ ]:
import pandas as pd
from datetime import datetime

class DataLoader:
    """A base class for loading data from a source."""
    
    def __init__(self, source):
        print("Parent DataLoader __init__ called.")
        self.source = source
        self._load_time = None
        
    def load_data(self):
        """The main method to load data. This will be implemented by child classes."""
        # The 'raise' keyword intentionally creates an error.
        # This forces any child class to create its own version of this method.
        raise NotImplementedError("Each subclass must implement its own load_data method.")
        
    def get_load_time(self):
        """A common utility method that all child classes will inherit."""
        if self._load_time:
            return self._load_time.strftime("%Y-%m-%d %H:%M:%S")
        return "Data has not been loaded yet."

This `DataLoader` class isn't very useful on its own. In fact, if you try to use its `load_data` method, it will give you an error. It's a blueprint, an abstract idea of what a data loader should be.

### Creating a Child Class: `CsvLoader`

Now, let's create a *specific* kind of `DataLoader` that knows how to read CSV files. 

To make `CsvLoader` a child of `DataLoader`, we put the parent class name in parentheses in the class definition.

In [ ]:
# CsvLoader is a child of DataLoader
class CsvLoader(DataLoader):
    """A specific DataLoader for CSV files."""
    
    def load_data(self):
        """This is the specialized implementation for CSVs."""
        print("Child CsvLoader: Reading data from a CSV source...")
        # In a real scenario, self.source would be a file path
        # For this example, we'll assume the source is the data itself
        from io import StringIO
        self._load_time = datetime.now()
        return pd.read_csv(StringIO(self.source))

# --- Usage ---
csv_data_string = "col_a,col_b\n1,A\n2,B"
my_csv_loader = CsvLoader(source=csv_data_string)

# It has its own 'load_data' method
df = my_csv_loader.load_data()
print("\nLoaded DataFrame:")
print(df)

# And it inherited the 'get_load_time' method from its parent!
print(f"\nData loaded at: {my_csv_loader.get_load_time()}")

Notice that we didn't have to write `get_load_time` again in `CsvLoader`. It inherited it for free from `DataLoader`! This is code reuse in action.

## Extending the Parent with `super()`

What if our child class needs its own `__init__` method to handle special attributes? For example, a `CsvLoader` might need to know the separator character (like a comma or a semicolon).

If you define a new `__init__` in the child, it **overrides** (replaces) the parent's `__init__`. The parent's constructor is no longer called automatically. This is a problem because the parent's `__init__` might do important setup work (like setting `self.source`).

The solution is to explicitly call the parent's `__init__` method from the child's `__init__` using `super()`.

In [ ]:
class CsvLoaderWithSeparator(DataLoader):
    """A CsvLoader that can handle different separators."""
    
    def __init__(self, source, separator=','):
        print("Child CsvLoaderWithSeparator __init__ called.")
        # This is the crucial step: Call the parent's constructor
        # to let it handle its part of the setup (the 'source').
        super().__init__(source)
        
        # Now, handle the child's specific attribute
        self.separator = separator
        
    # This method overrides the parent's placeholder method
    def load_data(self):
        print(f"Child CsvLoaderWithSeparator: Reading data with separator '{self.separator}'...")
        from io import StringIO
        self._load_time = datetime.now()
        return pd.read_csv(StringIO(self.source), sep=self.separator)

# --- Usage ---
tsv_data_string = "col_a;col_b\n1;A\n2;B" # Semicolon separated
my_tsv_loader = CsvLoaderWithSeparator(source=tsv_data_string, separator=';')

df_tsv = my_tsv_loader.load_data()
print("\nLoaded DataFrame:")
print(df_tsv)

# We can still access attributes set by the parent's __init__
print(f"\nSource: {my_tsv_loader.source}")
# And methods from the parent
print(f"Load time: {my_tsv_loader.get_load_time()}")

**Rule of thumb:** If your child class has its own `__init__`, you should almost always call `super().__init__(...)` as the first line inside it.

## Real-World Usage for Data Scientists

Inheritance is everywhere in data science libraries. It provides a framework for extension.

- **Scikit-learn Estimators**: At the heart of scikit-learn is a base class called `BaseEstimator`. Every single model and transformer in the library (e.g., `LinearRegression`, `StandardScaler`, `RandomForestClassifier`) inherits from it. This ensures they all share a consistent interface, including methods like `get_params()` and `set_params()`. They also inherit from other base classes like `ClassifierMixin` or `RegressorMixin`, which provide default implementations for methods like `score()`.

- **Custom Transformers**: When you build a custom data cleaning step for a scikit-learn `Pipeline`, you create a class that inherits from `BaseEstimator` and `TransformerMixin`. This gives your custom class the `fit_transform` method for free and makes it compatible with the entire scikit-learn ecosystem.

Let's create another child class for JSON data to see how clean this pattern is.

In [ ]:
class JsonLoader(DataLoader):
    """A specific DataLoader for JSON files (in 'records' orientation)."""
    
    def __init__(self, source):
        print("Child JsonLoader __init__ called.")
        super().__init__(source)
        
    def load_data(self):
        """Specialized implementation for JSON."""
        print("Child JsonLoader: Reading data from a JSON source...")
        self._load_time = datetime.now()
        # The 'orient' parameter is specific to JSON loading
        return pd.read_json(self.source, orient='records')

# --- Usage ---
json_data_string = '[{"col_a": 1, "col_b": "A"}, {"col_a": 2, "col_b": "B"}]'
my_json_loader = JsonLoader(source=json_data_string)

df_json = my_json_loader.load_data()
print("\nLoaded DataFrame:")
print(df_json)

print(f"\nLoad time: {my_json_loader.get_load_time()}")

We now have two specialized loader classes, `CsvLoaderWithSeparator` and `JsonLoader`, that both build upon the common foundation of `DataLoader`. We didn't have to duplicate the logic for handling `source` or `_load_time`.

## Common Mistakes and Misconceptions

1.  **Forgetting to call `super().__init__`**: This is the most common mistake. If the parent `__init__` sets up important attributes, forgetting to call it will cause errors later when the child tries to access those attributes.

2.  **Using the wrong relationship**: Inheritance implies an **"is-a"** relationship. A `CsvLoader` *is a* `DataLoader`. Sometimes, a **"has-a"** relationship is more appropriate. For example, a `Car` *has a* `Engine`. You wouldn't make `Car` inherit from `Engine`. Instead, you would create an `Engine` object as an attribute of the `Car` object: `self.engine = Engine()`. This is called **Composition** and is another powerful way to combine classes.

3.  **Overriding a method by accident**: If you create a method in a child class with the same name as a parent method, the child's method will always be used. This is usually what you want (it's called overriding), but if you're not careful, you can do it by mistake and wonder why the parent's logic isn't running.

## Recap

- **Inheritance**: A mechanism for a new class (child) to receive the attributes and methods of an existing class (parent).
- **"is-a" Relationship**: The core idea of inheritance. A `Car` is a `Vehicle`.
- **Syntax**: `class ChildClass(ParentClass):`
- **`super()`**: A built-in function used to call methods from the parent class. It's essential for calling the parent's `__init__` from the child's `__init__`.
- **Method Overriding**: A child class can provide its own specific implementation of a method that it inherited from the parent class.
- Inheritance is a cornerstone of object-oriented design, promoting code reuse and logical hierarchies.

## Suggested Next Lesson

We've seen that different child classes (`CsvLoader`, `JsonLoader`) can have their own versions of the `load_data` method. What if we could treat all these different loader objects in the same way, calling `load_data` on them without needing to know their specific type? This powerful concept, where objects of different classes can be treated as objects of a common superclass, is called **Polymorphism**. It's the next key to unlocking flexible and scalable OOP design.